In [ ]:
# Imports & Configuration
from pathlib import Path

import duckdb
import polars as pl
from great_tables import GT

# Change only this path
SOURCE = Path("../../raw_data/raw_fx_rates.csv").resolve().as_posix()

# DuckDB connection
con = duckdb.connect()

In [ ]:
# ============================================================
# Functions
# ============================================================

def show_sql(query: str, max_rows: int = 100):
    """Execute DuckDB SQL and display results using Great Tables."""
    result = con.sql(query).limit(max_rows)

    df = pl.DataFrame(
        result.fetchall(),
        schema=result.columns,
        orient="row"
    )

    display(GT(df))


In [ ]:
# 01 | Source Overview

show_sql(f"""
    SUMMARIZE
    SELECT *
    FROM read_csv('{SOURCE}', all_varchar = true)
""")

In [ ]:
# 02 | Data Quality Checks

show_sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) - COUNT(DISTINCT (date, currency))
            AS duplicate_rates,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(date), '') IS NULL
               OR NULLIF(TRIM(currency), '') IS NULL
               OR NULLIF(TRIM(fx_rate), '') IS NULL
        ) AS missing_required,

        COUNT(*) FILTER (
            WHERE TRY_CAST(date AS DATE) IS NULL
        ) AS invalid_dates,

        COUNT(*) FILTER (
            WHERE TRY_CAST(fx_rate AS DECIMAL(18,6)) IS NULL
               OR TRY_CAST(fx_rate AS DECIMAL(18,6)) <= 0
        ) AS invalid_rates,

        COUNT(*) FILTER (
            WHERE currency NOT IN ('EUR', 'HUF')
        ) AS unexpected_currency

    FROM read_csv('{SOURCE}', all_varchar = true)
""")

In [ ]:
# 03 | FX Date Coverage

show_sql(f"""
    WITH dates AS (
        SELECT
            currency,
            CAST(date AS DATE) AS rate_date
        FROM read_csv('{SOURCE}', all_varchar = true)
    )
    SELECT
        currency,
        MIN(rate_date) AS first_date,
        MAX(rate_date) AS last_date,
        COUNT(*) AS available_rates,
        DATE_DIFF('day', MIN(rate_date), MAX(rate_date)) + 1
            AS calendar_days,
        DATE_DIFF('day', MIN(rate_date), MAX(rate_date)) + 1 - COUNT(*)
            AS days_without_rates
    FROM dates
    GROUP BY currency
    ORDER BY currency
""")

In [ ]:
# 04 | FX Coverage Against Orders

show_sql(f"""
    WITH orders AS (
        SELECT
            CAST(created_at AS TIMESTAMPTZ)
                AT TIME ZONE 'Europe/Prague' AS local_datetime,
            currency
        FROM read_csv('{SOURCE.replace("raw_fx_rates.csv", "raw_orders_sk.csv")}', all_varchar = true)

        UNION ALL

        SELECT
            CAST(created_at AS TIMESTAMPTZ)
                AT TIME ZONE 'Europe/Prague',
            currency
        FROM read_csv('{SOURCE.replace("raw_fx_rates.csv", "raw_orders_hu.csv")}', all_varchar = true)
    ),
    fx AS (
        SELECT
            CAST(date AS DATE) AS rate_date,
            currency
        FROM read_csv('{SOURCE}', all_varchar = true)
    )
    SELECT
        o.currency,
        COUNT(*) AS order_lines,
        COUNT(*) FILTER (
            WHERE NOT EXISTS (
                SELECT 1
                FROM fx
                WHERE fx.currency = o.currency
                  AND fx.rate_date <= CAST(o.local_datetime AS DATE)
            )
        ) AS missing_historical_fx
    FROM orders o
    GROUP BY o.currency
    ORDER BY o.currency
""")

| Area | Finding | Recommendation |
|---|---|---|
| Uniqueness | No duplicate date × currency records | Preserve source grain |
| Data quality | No missing or invalid values detected | Apply appropriate data types |
| Date coverage | 191 rates per currency, 85 calendar days without rates | Use the latest available historical rate |
| Order coverage | All 8,540 raw SK/HU order lines have historical FX coverage | Apply an as-of FX lookup in the business layer |
| Currency | EUR and HUF | No conversion required for CZK |

**Staging:** Preserve original FX rates and dates.

**Business logic:** Convert sales to CZK using the most recent applicable rate on or before the Prague reporting date.